# 07 — SQL Window Functions

### Why this exists

Window functions let you compare a row with related rows without collapsing the result. They are useful for ranking, running totals and change detection.

### Learning objectives

- distinguish GROUP BY from windowing
- build a partitioned ranking
- build a running or lagged metric

### Prerequisite

06 — SQL Joins and Aggregations

### Engineering loop

**Why → Sketch → Predict → Build → Measure → Break → Diagnose → Improve → Production**


## Capability contract

**Capability:** C02 — Data Access Querying & Modeling  
**Workstream:** Revenue Data Platform  
**Primary roles:** Data Engineer|Analytics Engineer|Platform Engineer  
**You will prove:** Translate a business question into correct data access and defend the model/query behavior.


## 🖊️ Sketch note

![Sketch note — SQL Window Functions](../assets/sketch-notes/07-sql-window-functions.svg)

> **Visual-first rule:** spend 30–60 seconds explaining the sketch in your own words before reading the implementation. Then return to it after the experiment and redraw it from memory.


## Agent-native lens

**Agent can do:** routine implementation or a first-pass analysis.

**You must understand:** the mechanism and its assumptions.

**You must verify:** use tests, measurements or adversarial data that could prove the output wrong.

**You must decide:** the trade-off, failure response or production boundary.


## Apprenticeship bridge

Use this notebook to prepare for the following Acme Commerce work scenario(s). The scenario gives you symptoms and constraints; it does **not** give you the diagnosis.

- **S01 — Ticket DATA-1042: Revenue dashboard mismatch** → [`../work_simulations/01_acme_commerce_revenue_mismatch/README.md`](../work_simulations/01_acme_commerce_revenue_mismatch/README.md)

**How to use the bridge:** finish the experiment and independent challenge here, then enter the ticket without reopening the notebook as an answer key. Bring your own prediction, evidence and verification plan.


## Predict before you run

Write down one thing you expect to observe and why. Do not use the output as your prediction.


## Production bridge

Ask: what happens when the data is 100× larger, arrives late, changes shape, or is queried by several teams? State the assumption that makes your experiment useful.


## Your task

**Goal:** Find each customer's first and latest order.

**Do this:** Write the SQL, state the row grain, and use window functions or another defensible approach. Test ties or another edge case.

**Before you finish, save:** your prediction, the key measurement, the failure/edge case you tested, your diagnosis, and the production implication.

**Done when:** another engineer can reproduce your result and understand why you made the decision.

### Exercise detail

## Independent challenge

Write a query that finds each customer’s first and most recent order.

### Evidence to keep

- your prediction
- one measurement
- the failure or edge case you tested
- one production implication


# 07 — SQL Window Functions

## The minimum theory

`GROUP BY` collapses many rows into summary rows.

A **window function** calculates across related rows while keeping the original rows.

```text
GROUP BY → many rows become one summary
WINDOW   → same rows + calculated columns
```

Windows are useful for rankings, running totals, previous/next values, rolling calculations, and deduplication.

`PARTITION BY` defines independent groups; `ORDER BY` defines sequence within each group.


In [ ]:
import duckdb,pandas as pd
orders=pd.DataFrame({"order_id":[1,2,3,4,5,6,7,8],"customer_id":[1,1,1,2,2,3,3,3],"order_date":pd.to_datetime(["2026-01-01","2026-01-05","2026-01-10","2026-01-02","2026-01-08","2026-01-01","2026-01-03","2026-01-09"]),"amount":[100,50,200,80,120,40,70,90]})
con=duckdb.connect(); con.register("orders",orders)


In [ ]:
con.sql("SELECT customer_id,order_id,order_date,amount,SUM(amount) OVER (PARTITION BY customer_id ORDER BY order_date,order_id) AS running_total FROM orders ORDER BY customer_id,order_date,order_id").df()


In [ ]:
con.sql("WITH ranked AS (SELECT *,ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY amount DESC,order_id) AS rn FROM orders) SELECT * FROM ranked WHERE rn<=2 ORDER BY customer_id,rn").df()


In [ ]:
con.sql("SELECT customer_id,order_id,order_date,amount,LAG(amount) OVER (PARTITION BY customer_id ORDER BY order_date,order_id) AS previous_amount FROM orders ORDER BY customer_id,order_date,order_id").df()


## Break it

Remove `PARTITION BY`, change ordering, create ties, and compare `ROW_NUMBER()` with `RANK()`.

### Production consideration
Window functions can require partitioning and sorting substantial data. Ask:

> What data must be partitioned/sorted, and can the query or data layout make that work manageable?

**Engineering challenge:** calculate daily sales, cumulative sales per country, previous-day sales, and day-over-day percentage change.


---

## Check your work

You have finished the exercise. **Now** compare your reasoning and evidence with the reference solution.

[Open the reference solution for Notebook 07](https://github.com/ankit-rathi/data-systems-lab/blob/main/solutions/notebooks/07.md)
